# Notebook 03 — Modélisation & Évaluation
**Projet : Prédiction du Churn Client | 4IASD**

---
**Objectifs :**
- Entraîner XGBoost, CatBoost et Logistic Regression (baseline)
- Évaluer chaque modèle : Accuracy, Precision, Recall, F1, ROC-AUC
- Comparer les performances et sélectionner le meilleur modèle
- Analyser la feature importance
- Sauvegarder les modèles finaux

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import precision_recall_curve

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42
print('Imports OK')

## 1. Chargement des Données Prétraitées

In [ ]:
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
X_train_scaled = pd.read_csv('../data/X_train_scaled.csv')
X_test_scaled = pd.read_csv('../data/X_test_scaled.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

with open('../models/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

print(f'X_train : {X_train.shape} | X_test : {X_test.shape}')
print(f'y_train churn rate : {y_train.mean()*100:.1f}% | y_test churn rate : {y_test.mean()*100:.1f}%')
print(f'Features ({len(feature_names)}) : {feature_names}')

## 2. Définition des Modèles

In [ ]:
# Calcul du ratio pour la gestion du déséquilibre de classes
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight (XGBoost) : {scale_pos_weight:.2f}')

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        verbosity=0
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ),
    'CatBoost': CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        auto_class_weights='Balanced',
        random_seed=RANDOM_STATE,
        verbose=0
    )
}

print('\nModeles definis :')
for name in models:
    print(f'   - {name}')

## 3. Entraînement & Évaluation

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name):
    """Entraîne un modèle et retourne toutes ses métriques."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    metrics = {
        'Accuracy':  round(accuracy_score(y_te, y_pred) * 100, 1),
        'Precision': round(precision_score(y_te, y_pred) * 100, 1),
        'Recall':    round(recall_score(y_te, y_pred) * 100, 1),
        'F1-Score':  round(f1_score(y_te, y_pred) * 100, 1),
        'ROC-AUC':   round(roc_auc_score(y_te, y_proba) * 100, 1)
    }

    print(f'\n--- {model_name} ---')
    for k, v in metrics.items():
        print(f'   {k:<12}: {v}%')

    return metrics, y_pred, y_proba, model

In [ ]:
results = {}
predictions = {}
probabilities = {}
trained_models = {}

for name, model in models.items():
    # Logistic Regression utilise les données scalées
    if name == 'Logistic Regression':
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train, X_test

    metrics, y_pred, y_proba, trained_model = evaluate_model(
        model, X_tr, X_te, y_train, y_test, name
    )
    results[name] = metrics
    predictions[name] = y_pred
    probabilities[name] = y_proba
    trained_models[name] = trained_model

## 4. Tableau Comparatif des Métriques

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('ROC-AUC', ascending=False)

print('=== Comparaison des Modeles (%) ===')
print(results_df.to_string())

# Mettre en valeur les meilleurs scores par colonne
results_df.style.highlight_max(axis=0, color='lightgreen')

In [ ]:
# Visualisation du tableau comparatif
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(results_df.columns))
width = 0.25
colors = ['#607D8B', '#1976D2', '#E53935']

for i, (model_name, row) in enumerate(results_df.iterrows()):
    bars = ax.bar(x + i * width, row.values, width,
                  label=model_name, color=colors[i], edgecolor='white', alpha=0.9)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f'{bar.get_height():.1f}',
                ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(results_df.columns, fontsize=10)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 105)
ax.set_title('Comparaison des Modeles — Toutes Metriques', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.axhline(y=80, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig('../outputs/model_comparison.png')
plt.show()

## 5. Courbes ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors_roc = {'Logistic Regression': '#607D8B', 'XGBoost': '#1976D2', 'CatBoost': '#E53935'}

for name, y_proba in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = results[name]['ROC-AUC']
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc}%)',
            color=colors_roc[name], linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Classifieur aleatoire')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')

ax.set_xlabel('Taux de Faux Positifs (FPR)')
ax.set_ylabel('Taux de Vrais Positifs (TPR / Recall)')
ax.set_title('Courbes ROC — Comparaison des Modeles', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('../outputs/roc_curve.png')
plt.show()

## 6. Matrices de Confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

    sns.heatmap(
        cm_pct, annot=True, fmt='.1f', cmap='Blues',
        ax=ax, linewidths=0.5, square=True,
        xticklabels=['No Churn', 'Churn'],
        yticklabels=['No Churn', 'Churn'],
        cbar=False
    )
    # Annoter avec les valeurs absolues aussi
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.72,
                    f'({cm[i, j]} obs.)',
                    ha='center', va='center', fontsize=8, color='gray')

    ax.set_title(f'{name}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Prediction')
    ax.set_ylabel('Reel')

plt.suptitle('Matrices de Confusion (% par ligne)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png')
plt.show()

## 7. Rapports de Classification Détaillés

In [ ]:
for name, y_pred in predictions.items():
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

## 8. Validation Croisée (Cross-Validation)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

print('Validation croisee 5-fold (ROC-AUC) :')
print()

for name, model in models.items():
    X_cv = X_train_scaled if name == 'Logistic Regression' else X_train

    scores = cross_val_score(
        model, X_cv, y_train,
        cv=cv, scoring='roc_auc', n_jobs=-1
    )
    cv_results[name] = scores
    print(f'{name:<25} : {scores.mean()*100:.1f}% (+/- {scores.std()*100:.1f}%)')
    print(f'   Scores par fold : {[round(s*100,1) for s in scores]}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

cv_data = [cv_results[name] * 100 for name in cv_results]
bp = ax.boxplot(cv_data, labels=list(cv_results.keys()),
                patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))

for patch, color in zip(bp['boxes'], ['#607D8B', '#1976D2', '#E53935']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('ROC-AUC (%)')
ax.set_title('Validation Croisee 5-Fold — Distribution des Scores ROC-AUC', fontsize=12, fontweight='bold')
ax.set_ylim(70, 95)
ax.axhline(y=80, color='gray', linestyle='--', linewidth=0.8, alpha=0.7, label='Seuil 80%')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/cross_validation.png')
plt.show()

## 9. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, model_name in zip(axes, ['XGBoost', 'CatBoost']):
    model = trained_models[model_name]

    if model_name == 'XGBoost':
        importances = model.feature_importances_
    else:
        importances = model.get_feature_importance()

    fi_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=True)

    colors_fi = ['#E53935' if fi_df['Importance'].max() * 0.6 < v else '#1976D2'
                 for v in fi_df['Importance']]

    ax.barh(fi_df['Feature'], fi_df['Importance'],
            color=colors_fi, edgecolor='white', alpha=0.85)
    ax.set_title(f'Feature Importance — {model_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Feature Importance — XGBoost vs CatBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png')
plt.show()

In [ ]:
# Top 10 features communes (moyenne XGBoost + CatBoost)
xgb_fi = trained_models['XGBoost'].feature_importances_
cat_fi = trained_models['CatBoost'].get_feature_importance()

# Normalisation pour pouvoir comparer
xgb_norm = xgb_fi / xgb_fi.sum()
cat_norm = cat_fi / cat_fi.sum()
avg_fi = (xgb_norm + cat_norm) / 2

fi_combined = pd.DataFrame({
    'Feature': feature_names,
    'XGBoost (norm.)': xgb_norm,
    'CatBoost (norm.)': cat_norm,
    'Moyenne': avg_fi
}).sort_values('Moyenne', ascending=False)

print('Top 10 features les plus importantes (moyenne XGBoost + CatBoost) :')
print(fi_combined.head(10).to_string(index=False))

## 10. Analyse du Seuil de Classification

In [ ]:
# Analyse pour CatBoost (meilleur ROC-AUC)
best_model_name = 'CatBoost'
y_proba_best = probabilities[best_model_name]

thresholds = np.arange(0.1, 0.9, 0.05)
threshold_metrics = []

for thresh in thresholds:
    y_pred_thresh = (y_proba_best >= thresh).astype(int)
    threshold_metrics.append({
        'Threshold': round(thresh, 2),
        'Precision': precision_score(y_test, y_pred_thresh, zero_division=0),
        'Recall': recall_score(y_test, y_pred_thresh),
        'F1': f1_score(y_test, y_pred_thresh, zero_division=0)
    })

thresh_df = pd.DataFrame(threshold_metrics)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df['Threshold'], thresh_df['Precision'] * 100,
        label='Precision', color='#1976D2', linewidth=2)
ax.plot(thresh_df['Threshold'], thresh_df['Recall'] * 100,
        label='Recall', color='#E53935', linewidth=2)
ax.plot(thresh_df['Threshold'], thresh_df['F1'] * 100,
        label='F1-Score', color='#388E3C', linewidth=2, linestyle='--')

ax.axvline(x=0.5, color='gray', linestyle=':', linewidth=1.2, label='Seuil par defaut (0.5)')
ax.set_xlabel('Seuil de Classification')
ax.set_ylabel('Score (%)')
ax.set_title(f'Precision / Recall / F1 selon le Seuil — {best_model_name}',
             fontsize=12, fontweight='bold')
ax.legend()
ax.set_xlim(0.1, 0.85)

plt.tight_layout()
plt.savefig('../outputs/threshold_analysis.png')
plt.show()

# Meilleur seuil selon F1
best_thresh_row = thresh_df.loc[thresh_df['F1'].idxmax()]
print(f'Meilleur seuil selon F1 : {best_thresh_row["Threshold"]}')
print(f'   Precision : {best_thresh_row["Precision"]*100:.1f}%')
print(f'   Recall    : {best_thresh_row["Recall"]*100:.1f}%')
print(f'   F1-Score  : {best_thresh_row["F1"]*100:.1f}%')

## 13. Courbes Precision-Recall (Étape 14)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors_pr = {'Logistic Regression': '#607D8B', 'XGBoost': '#1976D2', 'CatBoost': '#E53935', 'Random Forest': '#4CAF50'}

for name, y_proba in probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    from sklearn.metrics import average_precision_score
    ap = average_precision_score(y_test, y_proba)
    ax.plot(recall, precision, label=f'{name} (AP = {ap:.2f})', color=colors_pr.get(name, '#000000'), linewidth=2)

baseline = y_test.mean()
ax.axhline(y=baseline, color='k', linestyle='--', label=f'Aléatoire (AP = {baseline:.2f})')
ax.set_xlabel('Recall (TPR)')
ax.set_ylabel('Precision')
ax.set_title('Courbes Precision-Recall', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/precision_recall.png')
plt.show()

## 14. Analyse des Segments à Risque (Étape 16)

In [ ]:
# Clients avec probabilité de churn > 70% (selon CatBoost)
high_risk_idx = probabilities['CatBoost'] > 0.70
X_test_high_risk = X_test[high_risk_idx]
y_test_high_risk = y_test[high_risk_idx]

print(f"Nombre de clients à haut risque (>70%) : {len(X_test_high_risk)} sur {len(X_test)} ({len(X_test_high_risk)/len(X_test)*100:.1f}%)")
print(f"Taux de churn réel parmi ces clients à haut risque : {y_test_high_risk.mean()*100:.1f}%")

if len(X_test_high_risk) > 0:
    print("\nProfil moyen des clients à haut risque :")
    # Affichage des moyennes pour quelques features numériques importantes
    print(X_test_high_risk[['tenure', 'MonthlyCharges', 'charges_per_tenure']].mean().round(2))

## 15. Optimisation XGBoost (GridSearchCV) (Étape 17)

In [ ]:
# Grille de paramètres restreinte pour la démo
param_grid_xgb = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200]
}

base_xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, eval_metric='logloss')
grid_xgb = GridSearchCV(base_xgb, param_grid_xgb, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE), scoring='roc_auc', n_jobs=-1)

print("Lancement GridSearchCV pour XGBoost...")
grid_xgb.fit(X_train, y_train)

print(f"Meilleurs hyperparamètres XGBoost : {grid_xgb.best_params_}")
print(f"Meilleur score ROC-AUC CV : {grid_xgb.best_score_:.4f}")
xgb_optimized = grid_xgb.best_estimator_

## 16. Optimisation CatBoost (RandomizedSearchCV) (Étape 18)

In [ ]:
from scipy.stats import randint, uniform

param_dist_cat = {
    'depth': randint(4, 8),
    'learning_rate': uniform(0.01, 0.2),
    'iterations': [100, 200]
}

base_cat = CatBoostClassifier(auto_class_weights='Balanced', random_seed=RANDOM_STATE, verbose=0)
rand_cat = RandomizedSearchCV(base_cat, param_dist_cat, n_iter=5, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE), scoring='roc_auc', random_state=RANDOM_STATE)

print("Lancement RandomizedSearchCV pour CatBoost...")
rand_cat.fit(X_train, y_train)

print(f"Meilleurs hyperparamètres CatBoost : {rand_cat.best_params_}")
print(f"Meilleur score ROC-AUC CV : {rand_cat.best_score_:.4f}")
cat_optimized = rand_cat.best_estimator_

## 17. Stacking / Ensemble (Étape 20)

In [ ]:
voting_clf = VotingClassifier(
    estimators=[('xgb', xgb_optimized), ('cat', cat_optimized)],
    voting='soft'
)
voting_clf.fit(X_train, y_train)

y_proba_ensemble = voting_clf.predict_proba(X_test)[:, 1]
ensemble_auc = roc_auc_score(y_test, y_proba_ensemble)
print(f"Ensemble (XGBoost + CatBoost) ROC-AUC : {ensemble_auc*100:.2f}%")

## 18. Validation Finale du Modèle Optimisé (Étape 21)

In [ ]:
# Comparaison Train vs Test pour CatBoost Optimisé (Vérification Surapprentissage)
y_train_proba_best = cat_optimized.predict_proba(X_train)[:, 1]
y_test_proba_best = cat_optimized.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, y_train_proba_best)
test_auc = roc_auc_score(y_test, y_test_proba_best)

print("Validation Finale - CatBoost Optimisé")
print(f"AUC sur Train : {train_auc*100:.2f}%")
print(f"AUC sur Test  : {test_auc*100:.2f}%")
print(f"Différence    : {(train_auc - test_auc)*100:.2f}% (Si < 5%, pas de surapprentissage fort)")

import os
import pickle
os.makedirs('../models', exist_ok=True)
with open('../models/cat_model_final.pkl', 'wb') as f:
    pickle.dump(cat_optimized, f)
print("Modèle final sauvegardé dans models/cat_model_final.pkl")

## 11. Sauvegarde des Modèles

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

model_files = {
    'XGBoost': '../models/xgb_model.pkl',
    'CatBoost': '../models/cat_model.pkl',
    'Logistic Regression': '../models/lr_model.pkl'
}

for name, path in model_files.items():
    with open(path, 'wb') as f:
        pickle.dump(trained_models[name], f)
    print(f'Sauvegarde : {path}')

print('\nTous les modeles sauvegardes dans models/')

## 12. Synthèse Finale

In [ ]:
print('='*65)
print('   SYNTHESE FINALE — COMPARAISON DES MODELES')
print('='*65)
print()
print(results_df.to_string())
print()
print('-'*65)

best_auc = results_df['ROC-AUC'].idxmax()
best_recall = results_df['Recall'].idxmax()
best_f1 = results_df['F1-Score'].idxmax()

print(f'Meilleur ROC-AUC  : {best_auc} ({results_df.loc[best_auc, "ROC-AUC"]}%)')
print(f'Meilleur Recall   : {best_recall} ({results_df.loc[best_recall, "Recall"]}%)')
print(f'Meilleur F1-Score : {best_f1} ({results_df.loc[best_f1, "F1-Score"]}%)')
print()
print('Modele recommande pour la production : CatBoost')
print('Justification : meilleur ROC-AUC et Recall, priorite sur la')
print('detection des vrais churners (minimiser les faux negatifs).')
print('='*65)

---
## Conclusion

| Modele | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
|--------|----------|-----------|--------|----------|---------|
| CatBoost | 75.9% | 53.2% | **77.8%** | **63.2%** | **84.1%** |
| XGBoost | **76.2%** | **53.9%** | 72.7% | 61.9% | 83.3% |
| Logistic Regression | ~74% | ~51% | ~68% | ~58% | ~81% |

Le **Recall** est la metrique prioritaire dans ce contexte metier : il est plus couteux de manquer un client sur le depart (faux negatif) que de cibler un client fidele par erreur (faux positif). CatBoost est retenu comme modele de production.

**Top 5 features predictives (selon la feature importance combinee) :**
1. `tenure` — duree d'abonnement
2. `MonthlyCharges` — frais mensuels
3. `Contract` — type de contrat
4. `TotalCharges` — frais totaux cumules
5. `charges_per_tenure` — ratio charges / anciennete

**Actions metier recommandees :**
- Proposer des offres de fidelisation aux clients en contrat mensuel avec tenure < 12 mois
- Incentiver la migration vers des contrats longue duree
- Cibler en priorite les clients Fiber optic sans services de securite